# Physical parameterization ablation on UIEB

Kaggle runner comparing RGB-only, unconstrained latent T/B, and Beer–Lambert-parameterized T/B on the fixed UIEB 800/90 protocol. The first 800 sorted pairs form the train/validation pool (720/80 per seed); the last 90 pairs are used only for final testing. Enable Internet and a GPU accelerator before running all cells. Progress is checkpointed after every epoch, so rerunning the notebook in the same session resumes incomplete runs automatically. Use `learnable_physics_uieb_kaggle_resume.ipynb` when the checkpoint is attached from an older Kaggle output.

In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import shutil
import subprocess
import sys

REPO = Path('/kaggle/working/underwater-image-enhancement')
BRANCH = 'learnable-physics-extractor'
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch', '--depth', '1',
        'https://github.com/heniath/underwater-image-enhancement.git', str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
# Preserve Kaggle's preinstalled CUDA/RAPIDS stack. The base image already
# provides Torch, Torchvision, NumPy, SciPy, OpenCV and scikit-image.
if importlib.util.find_spec('kornia') is None:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.7.3'
    ], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'
], check=True)
# Make the src-layout package immediately importable in this already-running
# notebook kernel; editable installs do not always refresh sys.path here.
SRC_DIR = str(REPO / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

commit = subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
models = subprocess.run(
    ['uwir-profile', '--list'], check=True, capture_output=True, text=True
).stdout
assert 'parameterized_physics_unet' in models, 'Branch does not contain the requested model'
print('Repository commit:', commit)

In [ ]:
import torch

raw_candidates = [
    Path('/kaggle/input/datasets/larjeck/uieb-dataset-raw/raw-890'),
    *Path('/kaggle/input').glob('**/raw-890'),
]
reference_candidates = [
    Path('/kaggle/input/datasets/larjeck/uieb-dataset-reference/reference-890'),
    *Path('/kaggle/input').glob('**/reference-890'),
]
RAW_DIR = next((path for path in raw_candidates if path.is_dir()), None)
REFERENCE_DIR = next((path for path in reference_candidates if path.is_dir()), None)
assert RAW_DIR is not None, 'UIEB raw-890 folder was not found'
assert REFERENCE_DIR is not None, 'UIEB reference-890 folder was not found'

# UIEBDataset expects both folders below one root. Kaggle stores this copy
# in two read-only datasets, so expose them through lightweight symlinks.
UIEB_ROOT = Path('/kaggle/working/UIEB')
UIEB_ROOT.mkdir(parents=True, exist_ok=True)
for name, target in [('raw-890', RAW_DIR), ('reference-890', REFERENCE_DIR)]:
    link = UIEB_ROOT / name
    if link.is_symlink() and link.resolve() != target.resolve():
        link.unlink()
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS > 0, 'Enable a Kaggle GPU accelerator'
SMOKE = False  # True: one epoch, one seed, for a quick preflight.
EPOCHS = 1 if SMOKE else 100
SEEDS = [0] if not SMOKE else [0]  # Full protocol: [0, 1, 2]
BATCH_SIZE = 4
CROP_SIZE = 256
WORKERS = 2
ABLATIONS = {
    'rgb_baseline': dict(model='unet_3ch', reconstruction=0.0, smoothness=0.0),
    'latent_no_reconstruction': dict(model='learnable_latent_unet', reconstruction=0.0, smoothness=0.0),
    'latent_reconstruction': dict(model='learnable_latent_unet', reconstruction=1.0, smoothness=0.0),
    'physical_no_reconstruction': dict(model='parameterized_physics_unet', reconstruction=0.0, smoothness=0.01),
    'physical_reconstruction': dict(model='parameterized_physics_unet', reconstruction=1.0, smoothness=0.01),
}
OUTPUT_ROOT = Path('/kaggle/working/physics_parameterization_ablation_uieb')
# The resume notebook sets RESUME_CHECKPOINT before running this notebook.
# A checkpoint inside <root>/<config>/seed_<n>/ also lets us recover the
# completed runs stored beside it. PREVIOUS_OUTPUT_ROOT may be set directly.
RESUME_CHECKPOINT = globals().get('RESUME_CHECKPOINT') or os.environ.get('UWIR_RESUME_CHECKPOINT')
PREVIOUS_OUTPUT_ROOT = globals().get('PREVIOUS_OUTPUT_ROOT') or os.environ.get('UWIR_PREVIOUS_OUTPUT_ROOT')
if RESUME_CHECKPOINT:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    assert RESUME_CHECKPOINT.is_file(), f'Resume checkpoint not found: {RESUME_CHECKPOINT}'
    if (PREVIOUS_OUTPUT_ROOT is None and RESUME_CHECKPOINT.parent.name.startswith('seed_')
            and RESUME_CHECKPOINT.parent.parent.name in ABLATIONS):
        PREVIOUS_OUTPUT_ROOT = RESUME_CHECKPOINT.parents[2]
if PREVIOUS_OUTPUT_ROOT:
    PREVIOUS_OUTPUT_ROOT = Path(PREVIOUS_OUTPUT_ROOT)
    assert PREVIOUS_OUTPUT_ROOT.is_dir(), f'Previous output not found: {PREVIOUS_OUTPUT_ROOT}'
    if PREVIOUS_OUTPUT_ROOT.resolve() != OUTPUT_ROOT.resolve():
        shutil.copytree(PREVIOUS_OUTPUT_ROOT, OUTPUT_ROOT, dirs_exist_ok=True)
        print('Copied previous output:', PREVIOUS_OUTPUT_ROOT, '->', OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda')
print('Raw images:', RAW_DIR)
print('References:', REFERENCE_DIR)
print('Combined dataset root:', UIEB_ROOT)
print('GPUs:', [torch.cuda.get_device_name(index) for index in range(NUM_GPUS)])
print('Mode:', 'smoke' if SMOKE else 'full', '| epochs:', EPOCHS, '| seeds:', SEEDS)
print('Ablations:', list(ABLATIONS))

In [ ]:
import json
import random
import time

import numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import Compose, ToTensor

from uwir.cli.train import EarlyStopping, load_ckpt, save_ckpt, train_epoch, val_loss_epoch
from uwir.data.datasets import UIEBDataset
from uwir.losses import CompositeLoss, PhysicsConsistentLoss
from uwir.metrics import evaluate_loader
from uwir.models import build_model
from uwir.training.schedulers import CosineAnnealingRestartLR

transform = Compose([ToTensor()])
train_base = UIEBDataset(
    str(UIEB_ROOT), transform=transform, augment=True, img_size=CROP_SIZE
)
eval_base = UIEBDataset(
    str(UIEB_ROOT), transform=transform, augment=False, img_size=CROP_SIZE
)
assert len(train_base) == 890, f'Expected 890 UIEB pairs, found {len(train_base)}'
TRAIN_VAL_INDICES = list(range(800))
TEST_INDICES = list(range(800, 890))
test_dataset = Subset(eval_base, TEST_INDICES)

def collate_rgb(batch):
    inputs = torch.stack([item[0] for item in batch])
    targets = torch.stack([item[1] for item in batch])
    return inputs, targets

def make_loader(dataset, *, shuffle, batch_size=BATCH_SIZE):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=WORKERS,
        pin_memory=True, drop_last=shuffle, collate_fn=collate_rgb,
        persistent_workers=WORKERS > 0,
    )

print('UIEB split: train/validation pool=800, held-out test=90')

In [ ]:
def _atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def _atomic_json_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2))
    os.replace(temporary, path)

def _checkpoint_payload(model, optimizer, scheduler, scaler, epoch, metrics, history,
                        best_psnr, best_ssim, early_stopping, complete=False):
    bare_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    return {
        'epoch': epoch, 'model': bare_model.state_dict(),
        'optimizer': optimizer.state_dict(), 'metrics': metrics,
        'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'history': history, 'best_psnr': best_psnr, 'best_ssim': best_ssim,
        'early_stopping': {
            'best': early_stopping.best, 'counter': early_stopping.counter,
            'stop': early_stopping.stop,
        },
        'complete': complete,
    }

def _run_is_complete(run_dir):
    last_path = run_dir / 'last_model.pth'
    if not last_path.is_file():
        return False
    checkpoint = torch.load(last_path, map_location='cpu', weights_only=False)
    return checkpoint.get('complete', True)

def train_run(config_name, config, seed):
    run_dir = OUTPUT_ROOT / config_name / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / 'best_model.pth'
    last_path = run_dir / 'last_model.pth'
    complete_path = run_dir / 'run_complete.json'
    if best_path.is_file() and (complete_path.is_file() or _run_is_complete(run_dir)):
        print(f'\n{config_name} seed {seed}: already complete; using {best_path}')
        return best_path

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    order = torch.randperm(800, generator=torch.Generator().manual_seed(seed)).tolist()
    val_indices = sorted(order[:80])
    train_indices = sorted(order[80:])
    train_loader = make_loader(Subset(train_base, train_indices), shuffle=True)
    val_loader = make_loader(Subset(eval_base, val_indices), shuffle=False)

    model = build_model(config['model']).to(DEVICE)
    supports_physics = getattr(model, 'supports_physics_loss', False)
    if NUM_GPUS > 1:
        model = torch.nn.DataParallel(model)
    enhancement_loss = CompositeLoss(
        lambda_l1=1.0, lambda_perc=1.0, lambda_ssim=0.0, device=DEVICE
    )
    criterion = (
        PhysicsConsistentLoss(
            enhancement_loss,
            lambda_reconstruction=config['reconstruction'],
            lambda_depth_smoothness=config['smoothness'],
        )
        if supports_physics else enhancement_loss
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingRestartLR(
        optimizer, periods=[EPOCHS], restart_weights=[1.0], eta_min=1e-6
    )
    scaler = torch.amp.GradScaler('cuda', enabled=True, init_scale=1024.0)

    split = {'seed': seed, 'train': train_indices, 'validation': val_indices, 'test': TEST_INDICES}
    _atomic_json_save(split, run_dir / 'split_manifest.json')
    history = {
        'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': [],
        'reconstruction': [], 'depth_smoothness': [], 'lr': [],
    }
    best_psnr = float('-inf')
    best_ssim = float('-inf')
    early_stopping = EarlyStopping(patience=20, min_delta=1e-4, mode='max')
    start_epoch = 1

    resume_path = last_path if last_path.is_file() else None
    if resume_path is None and RESUME_CHECKPOINT:
        candidate = Path(RESUME_CHECKPOINT)
        if candidate.parent.name == f'seed_{seed}' and candidate.parent.parent.name == config_name:
            resume_path = candidate
    if resume_path is None and best_path.is_file():
        resume_path = best_path
    if resume_path is not None:
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        bare_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        bare_model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        resumed_epoch = int(checkpoint['epoch'])
        if 'scheduler' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler'])
        else:
            scheduler.step(resumed_epoch)
        if 'scaler' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler'])
        history = checkpoint.get('history', history)
        best_metrics = torch.load(best_path, map_location='cpu', weights_only=False).get('metrics', {}) if best_path.is_file() else {}
        best_psnr = checkpoint.get('best_psnr', best_metrics.get('psnr', checkpoint.get('metrics', {}).get('psnr', float('-inf'))))
        best_ssim = checkpoint.get('best_ssim', best_metrics.get('ssim', checkpoint.get('metrics', {}).get('ssim', float('-inf'))))
        stopping_state = checkpoint.get('early_stopping', {})
        early_stopping.best = stopping_state.get('best', best_psnr)
        early_stopping.counter = stopping_state.get('counter', 0)
        early_stopping.stop = False
        start_epoch = resumed_epoch + 1
        print(f'\n{config_name} seed {seed}: resuming {resume_path} after epoch {resumed_epoch}')

    started = time.time()
    print(
        f'\n{config_name} | model={config["model"]} | seed={seed} | '
        f'train=720 validation=80 epochs={EPOCHS}'
    )
    if start_epoch > EPOCHS:
        _atomic_json_save({'epoch': start_epoch - 1, 'reason': 'epochs_complete'}, complete_path)
        return best_path

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_started = time.time()
        train_loss, parts = train_epoch(
            model, train_loader, optimizer, criterion, DEVICE, scaler=scaler
        )
        validation_loss = val_loss_epoch(
            model, val_loader, criterion, DEVICE, amp_enabled=True
        )
        metrics, _ = evaluate_loader(model, val_loader, DEVICE)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(validation_loss)
        history['val_psnr'].append(metrics['psnr'])
        history['val_ssim'].append(metrics['ssim'])
        history['reconstruction'].append(parts.get('reconstruction', 0.0))
        history['depth_smoothness'].append(parts.get('depth_smoothness', 0.0))
        history['lr'].append(lr)

        improved = (metrics['psnr'], metrics['ssim']) > (best_psnr, best_ssim)
        if improved:
            best_psnr, best_ssim = metrics['psnr'], metrics['ssim']
            save_ckpt(model, optimizer, epoch, {**metrics, 'val_loss': validation_loss}, str(best_path))
        stopped = early_stopping(metrics['psnr'])
        payload = _checkpoint_payload(
            model, optimizer, scheduler, scaler, epoch, metrics, history,
            best_psnr, best_ssim, early_stopping, complete=stopped or epoch == EPOCHS,
        )
        _atomic_torch_save(payload, last_path)
        _atomic_json_save(history, run_dir / 'training_history.json')
        if epoch == 1 or epoch % 5 == 0 or improved:
            elapsed = time.time() - epoch_started
            marker = ' BEST' if improved else ''
            print(
                f'{epoch:03d}/{EPOCHS} train={train_loss:.4f} val={validation_loss:.4f} '
                f'PSNR={metrics["psnr"]:.3f} SSIM={metrics["ssim"]:.4f} '
                f'recon={parts.get("reconstruction", 0.0):.4f} '
                f'smooth={parts.get("depth_smoothness", 0.0):.4f} '
                f'lr={lr:.2e} {elapsed:.1f}s{marker}'
            )
        if stopped:
            print('Early stopping at epoch', epoch)
            break

    _atomic_json_save({'epoch': epoch, 'reason': 'early_stop' if stopped else 'epochs_complete'}, complete_path)
    print(f'{config_name} seed {seed} finished in {(time.time() - started) / 60:.1f} min')
    return best_path

In [ ]:
best_checkpoints = {}
for config_name, config in ABLATIONS.items():
    best_checkpoints[config_name] = {}
    for seed in SEEDS:
        best_checkpoints[config_name][seed] = train_run(config_name, config, seed)
        gc.collect()
        torch.cuda.empty_cache()
best_checkpoints

In [ ]:
test_loader = make_loader(test_dataset, shuffle=False)
test_results = {}
for config_name, config in ABLATIONS.items():
    test_results[config_name] = {}
    for seed, checkpoint in best_checkpoints[config_name].items():
        model = build_model(config['model']).to(DEVICE)
        epoch, stored_metrics = load_ckpt(str(checkpoint), model, device=str(DEVICE))
        metrics, count = evaluate_loader(model, test_loader, DEVICE)
        test_results[config_name][str(seed)] = {
            'model': config['model'], 'reconstruction_weight': config['reconstruction'],
            'smoothness_weight': config['smoothness'], 'checkpoint': str(checkpoint),
            'epoch': epoch, 'count': count, 'validation': stored_metrics, 'test': metrics,
        }
        print(
            f'{config_name} | seed {seed} | epoch {epoch} | test n={count} | '
            f'PSNR={metrics["psnr"]:.3f} SSIM={metrics["ssim"]:.4f} '
            f'CIEDE2000={metrics["ciede2000"]:.3f}'
        )
(OUTPUT_ROOT / 'test_results.json').write_text(json.dumps(test_results, indent=2))

metric_names = ['psnr', 'ssim', 'ciede2000', 'uciqe', 'uiqm']
summary = {}
for config_name, runs in test_results.items():
    summary[config_name] = {}
    for metric in metric_names:
        values = np.array([run['test'][metric] for run in runs.values()])
        summary[config_name][metric] = {
            'mean': float(values.mean()),
            'std': float(values.std(ddof=1)) if len(values) > 1 else 0.0,
        }
    row = summary[config_name]
    print(
        f'{config_name:28s} PSNR={row["psnr"]["mean"]:.3f} '
        f'SSIM={row["ssim"]["mean"]:.4f} '
        f'CIEDE2000={row["ciede2000"]["mean"]:.3f}'
    )
(OUTPUT_ROOT / 'ablation_summary.json').write_text(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt

sample_input, sample_target, *_ = eval_base[TEST_INDICES[0]]
model = build_model('parameterized_physics_unet').to(DEVICE).eval()
physical_checkpoint = best_checkpoints['physical_reconstruction'][SEEDS[0]]
load_ckpt(str(physical_checkpoint), model, device=str(DEVICE))
with torch.no_grad():
    physics = model(sample_input.unsqueeze(0).to(DEVICE), return_physics=True)

def image(tensor):
    return tensor.detach().cpu().squeeze(0).permute(1, 2, 0).numpy().clip(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
panels = [
    ('Input', image(sample_input.unsqueeze(0)), None),
    ('Enhanced J', image(physics.enhanced), None),
    ('Reference', image(sample_target.unsqueeze(0)), None),
    ('Reconstructed I', image(physics.reconstructed), None),
    ('Background Bλ', image(physics.background), None),
    ('T red', physics.transmission[0, 0].cpu(), 'magma'),
    ('T green', physics.transmission[0, 1].cpu(), 'magma'),
    ('T blue', physics.transmission[0, 2].cpu(), 'magma'),
    ('Depth d(x)', physics.depth[0, 0].cpu(), 'viridis'),
    ('|J − reference|', np.abs(image(physics.enhanced) - image(sample_target.unsqueeze(0))).mean(2), 'inferno'),
]
for axis, (title, panel, cmap) in zip(axes.flat, panels):
    axis.imshow(panel, cmap=cmap, vmin=0, vmax=1)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
figure_path = OUTPUT_ROOT / 'physics_maps.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('Outputs:', OUTPUT_ROOT)